# Segmentazione cellule CZI con CellposeSAM

Questo notebook segmenta immagini in formato `.czi` usando CellposeSAM (Cellpose 4.x) e Google Colab.

## Istruzioni
1. Vai su **Runtime → Change runtime type → T4 GPU**
2. Esegui le celle in ordine
3. Carica la tua immagine `.czi` quando richiesto
4. Scarica i risultati alla fine

## 1. Installazione dipendenze

In [ ]:
# Installa le librerie necessarie
%pip install cellpose aicsimageio[czi] scikit-image shapely geojson matplotlib tifffile -q

# Verifica GPU
import torch
print(f"GPU disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Attenzione: nessuna GPU trovata. Verifica di aver selezionato T4 GPU in Runtime.")

## 2. Carica l'immagine CZI

In [ ]:
from google.colab import files
import io

print("Carica il tuo file .czi")
uploaded = files.upload()

czi_filename = list(uploaded.keys())[0]
print(f"File caricato: {czi_filename}")

## 3. Leggi e visualizza l'immagine

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aicsimageio import AICSImage

# Leggi il file CZI
aics_img = AICSImage(czi_filename)
img_data = aics_img.get_image_data("YX", T=0, Z=0, C=0)

print(f"Dimensioni immagine: {img_data.shape}")
print(f"Tipo dati: {img_data.dtype}")
print(f"Valore min: {img_data.min()}, max: {img_data.max()}")

# Visualizza l'immagine originale
plt.figure(figsize=(10, 8))
plt.imshow(img_data, cmap='gray')
plt.title('Immagine originale')
plt.colorbar()
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Normalizzazione immagine

In [ ]:
from skimage import exposure

# Normalizza tra 0 e 255 (necessario per Cellpose)
def normalize_image(img, percentile_low=1, percentile_high=99):
    p_low = np.percentile(img, percentile_low)
    p_high = np.percentile(img, percentile_high)
    img_norm = np.clip(img, p_low, p_high)
    img_norm = ((img_norm - p_low) / (p_high - p_low) * 255).astype(np.uint8)
    return img_norm

img_normalized = normalize_image(img_data)

print(f"Immagine normalizzata: min={img_normalized.min()}, max={img_normalized.max()}")

plt.figure(figsize=(10, 8))
plt.imshow(img_normalized, cmap='gray')
plt.title('Immagine normalizzata')
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Segmentazione con CellposeSAM

Parametri principali:
- `diameter`: diametro medio delle cellule in pixel. Metti `None` per rilevarlo automaticamente
- `channels`: `[0, 0]` per immagini in scala di grigi
- `flow_threshold`: soglia per i flow (default 0.4, abbassa se mancano cellule)
- `cellprob_threshold`: soglia probabilità cellula (default 0.0, abbassa per trovare più cellule)

In [ ]:
from cellpose import models

# -------------------------------------------------------
# PARAMETRI DA CONFIGURARE
DIAMETER = None          # None = rilevamento automatico, oppure es. 30 (pixel)
FLOW_THRESHOLD = 0.4     # abbassa (es. 0.2) se mancano cellule
CELLPROB_THRESHOLD = 0.0 # abbassa (es. -1.0) se mancano cellule
# -------------------------------------------------------

# Carica il modello CellposeSAM (cpsam)
gpu_available = torch.cuda.is_available()
model = models.CellposeModel(model_type='cpsam', gpu=gpu_available)

print(f"Modello caricato: cpsam")
print(f"Uso GPU: {gpu_available}")
print("Avvio segmentazione...")

# Esegui la segmentazione
masks, flows, styles = model.eval(
    img_normalized,
    diameter=DIAMETER,
    channels=[0, 0],
    flow_threshold=FLOW_THRESHOLD,
    cellprob_threshold=CELLPROB_THRESHOLD
)

n_cells = len(np.unique(masks)) - 1  # escludi il background (0)
print(f"\nSegmentazione completata!")
print(f"Cellule trovate: {n_cells}")

## 6. Visualizza i risultati

In [ ]:
from cellpose import plot

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Immagine originale
axes[0].imshow(img_normalized, cmap='gray')
axes[0].set_title('Immagine originale')
axes[0].axis('off')

# Maschera con colori casuali per cellula
axes[1].imshow(masks, cmap='nipy_spectral')
axes[1].set_title(f'Maschere ({n_cells} cellule)')
axes[1].axis('off')

# Overlay contorni sull'immagine originale
from skimage.segmentation import find_boundaries
boundaries = find_boundaries(masks, mode='outer')
overlay = np.stack([img_normalized]*3, axis=-1)  # RGB
overlay[boundaries] = [255, 0, 0]  # contorni in rosso
axes[2].imshow(overlay)
axes[2].set_title('Contorni sovrapposti')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('segmentation_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salvata: segmentation_result.png")

## 7. Estrai i poligoni e salva in GeoJSON

In [ ]:
from skimage import measure
import geojson
import json

features = []
cell_stats = []

for cell_id in np.unique(masks)[1:]:  # salta background (0)
    cell_mask = masks == cell_id

    # Estrai il contorno
    contours = measure.find_contours(cell_mask, 0.5)
    if not contours:
        continue

    # Prendi il contorno più lungo (cellula principale)
    contour = max(contours, key=len)

    # Converti in formato [x, y] (GeoJSON usa x=colonna, y=riga)
    polygon_coords = [[float(c[1]), float(c[0])] for c in contour]
    # Chiudi il poligono (GeoJSON richiede che primo == ultimo punto)
    polygon_coords.append(polygon_coords[0])

    # Calcola proprietà della cellula
    props = measure.regionprops(cell_mask.astype(int))[0]
    area = props.area
    perimeter = props.perimeter
    centroid_y, centroid_x = props.centroid
    circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

    cell_stats.append({
        'id': int(cell_id),
        'area_px': area,
        'perimeter_px': perimeter,
        'circularity': circularity,
        'centroid_x': centroid_x,
        'centroid_y': centroid_y
    })

    # Crea feature GeoJSON
    feature = geojson.Feature(
        geometry=geojson.Polygon([polygon_coords]),
        properties={
            'cell_id': int(cell_id),
            'area_px': float(area),
            'perimeter_px': float(perimeter),
            'circularity': float(circularity),
            'centroid_x': float(centroid_x),
            'centroid_y': float(centroid_y)
        }
    )
    features.append(feature)

# Salva GeoJSON
feature_collection = geojson.FeatureCollection(features)
with open('cells.geojson', 'w') as f:
    geojson.dump(feature_collection, f, indent=2)

print(f"GeoJSON salvato: cells.geojson")
print(f"Cellule esportate: {len(features)}")

## 8. Salva la maschera binaria

In [ ]:
import tifffile

# Maschera binaria (0 = sfondo, 255 = cellula)
binary_mask = (masks > 0).astype(np.uint8) * 255
tifffile.imwrite('binary_mask.tif', binary_mask)
print("Maschera binaria salvata: binary_mask.tif")

# Maschera con ID per ogni cellula (utile per analisi successive)
tifffile.imwrite('instance_mask.tif', masks.astype(np.uint16))
print("Maschera istanze salvata: instance_mask.tif")

# Visualizza maschera binaria
plt.figure(figsize=(10, 8))
plt.imshow(binary_mask, cmap='gray')
plt.title('Maschera binaria')
plt.axis('off')
plt.tight_layout()
plt.show()

## 9. Statistiche e CSV

In [ ]:
import pandas as pd

df = pd.DataFrame(cell_stats)
print(df.describe())

df.to_csv('cell_measurements.csv', index=False)
print(f"\nCSV salvato: cell_measurements.csv")

# Istogramma aree
plt.figure(figsize=(8, 4))
plt.hist(df['area_px'], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Area (pixel)')
plt.ylabel('Numero cellule')
plt.title('Distribuzione aree cellule')
plt.tight_layout()
plt.savefig('area_distribution.png', dpi=150)
plt.show()

## 10. Scarica tutti i risultati

In [ ]:
import zipfile
from google.colab import files

# Comprimi tutti i file di output
output_files = [
    'segmentation_result.png',
    'binary_mask.tif',
    'instance_mask.tif',
    'cells.geojson',
    'cell_measurements.csv',
    'area_distribution.png'
]

with zipfile.ZipFile('risultati_segmentazione.zip', 'w') as zf:
    for f in output_files:
        try:
            zf.write(f)
            print(f"Aggiunto: {f}")
        except FileNotFoundError:
            print(f"Non trovato (saltato): {f}")

print("\nScaricamento ZIP...")
files.download('risultati_segmentazione.zip')